In [6]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Load data
df_filled = pd.read_csv("DC_Master_ARIMA_Filled.csv", parse_dates=['Date'], index_col='Date')

df_filled = df_filled[df_filled.index <= pd.to_datetime("2024-10-01")]

print(df_filled.tail())

            House_Index       CPI  Poverty_Rate  Unemployment_Rate  \
Date                                                                 
2023-10-01       371.05  308.8365     10.926301                3.6   
2024-01-01       377.13  309.9610     10.870993                3.9   
2024-04-01       392.17  314.3875     10.835202                3.4   
2024-07-01       394.52  316.4450     10.800021                4.6   
2024-10-01       392.54  317.0635     10.765442                4.1   

            Median_Household_Income   Population  Interest_Rate  \
Date                                                              
2023-10-01            112308.235946  6368.533000           5.33   
2024-01-01            112690.097338  6391.185000           5.33   
2024-04-01            113086.006176  6413.837000           5.33   
2024-07-01            113494.043541  6436.489000           5.33   
2024-10-01            113912.552644  6457.275841           4.83   

            Mortgage_Rate        GDP  


In [8]:
# --- Stationarity Test Function ---
def test_stationarity(timeseries, var_name, regression_kpss='c'):
    """Performs ADF and KPSS tests and prints results."""
    print(f"\n--- Stationarity Tests for {var_name} ---")
    stationary = False # Flag to track stationarity

    # Drop NaNs that might result from differencing
    timeseries = timeseries.dropna()
    # --- ADF Test ---
    try:
        # Null Hypothesis: Series is non-stationary (has a unit root)
        adf_result = adfuller(timeseries, autolag='AIC')
        adf_stat = adf_result[0]
        adf_pvalue = adf_result[1]
        print(f"ADF Statistic: {adf_stat:.3f}")
        print(f"ADF p-value: {adf_pvalue:.3f}")
        adf_stationary = adf_pvalue < 0.05
        if adf_stationary:
            print("ADF Result: Likely Stationary (reject H0)")
        else:
            print("ADF Result: Likely Non-Stationary (fail to reject H0)")
    except Exception as e:
        print(f"ADF Test Failed: {e}")
        adf_stationary = False # Treat failure as non-stationary indication

    # --- KPSS Test ---
    try:
        # Null Hypothesis: Series is stationary (around a constant level or trend)
        kpss_result = kpss(timeseries, regression=regression_kpss, nlags="auto")
        kpss_stat = kpss_result[0]
        kpss_pvalue = kpss_result[1]
        print(f"\nKPSS Statistic ({regression_kpss}): {kpss_stat:.3f}")
        print(f"KPSS p-value ({regression_kpss}): {kpss_pvalue:.3f}")
        # If p < critical value (e.g., 0.05), reject H0 (suggests non-stationarity)
        kpss_stationary = kpss_pvalue >= 0.05
        if kpss_stationary:
                print(f"KPSS Result ({regression_kpss}): Likely Stationary (fail to reject H0)")
        else:
                print(f"KPSS Result ({regression_kpss}): Likely Non-Stationary (reject H0)")
    except Exception as e:
        print(f"KPSS Test Failed ({regression_kpss}): {e}")
        kpss_stationary = False # Treat failure cautiously

    # Combined Interpretation
    if adf_stationary and kpss_stationary:
            print("Combined: Likely Stationary")
            stationary = True
    elif not adf_stationary and not kpss_stationary:
            print("Combined: Likely Non-Stationary (Unit Root)")
            stationary = False
    else: # Conflicting results
            print("Combined: Results are conflicting or inconclusive. Examine plots.")
            # Be conservative: if either test suggests non-stationarity, assume non-stationary
            stationary = False

    return stationary # Return the stationarity flag

# --- Test Variables ---
integration_orders = {}
numeric_cols = df_filled.select_dtypes(include=['number']).columns

for col in numeric_cols:
    print(f"\n{'='*15} Testing Variable: {col} {'='*15}")

    # Test original series (level) - check for stationarity around constant ('c')
    # If it looks strongly trended, one might also check 'ct', but let's start with 'c'
    is_i0 = test_stationarity(df_filled[col], f"{col} (Original)", regression_kpss='c')
    if is_i0:
        integration_orders[col] = 'I(0)'
        print(f"Conclusion for {col}: Likely I(0)")
        print("-" * 50)
        continue # Skip differencing if already stationary

    # Test first difference
    diff1_series = df_filled[col].diff()
    is_i1 = test_stationarity(diff1_series, f"{col} (1st Difference)", regression_kpss='c')
    if is_i1:
        integration_orders[col] = 'I(1)'
        print(f"Conclusion for {col}: Likely I(1)")
        print("-" * 50)
        continue # Stop if first difference is stationary

    # Test second difference (only if first difference wasn't stationary)
    print(f"\nFirst difference of {col} appears non-stationary, testing 2nd difference...")
    diff2_series = df_filled[col].diff().diff()
    is_i2 = test_stationarity(diff2_series, f"{col} (2nd Difference)", regression_kpss='c')
    if is_i2:
        integration_orders[col] = 'I(2)'
        print(f"Conclusion for {col}: Likely I(2)")
    else:
        integration_orders[col] = 'I(>2) or Complex'
        print(f"Conclusion for {col}: Requires more than 2 differences or has complex structure.")
    print("-" * 50)

# --- Summarize Integration Orders ---
print("\n\n--- Summary of Integration Orders ---")
for var, order in integration_orders.items():
    print(f"{var}: {order}")


=============== Testing Variable: House_Index ===============

--- Stationarity Tests for House_Index (Original) ---
ADF Statistic: -0.199
ADF p-value: 0.939
ADF Result: Likely Non-Stationary (fail to reject H0)

KPSS Statistic (c): 1.835
KPSS p-value (c): 0.010
KPSS Result (c): Likely Non-Stationary (reject H0)
Combined: Likely Non-Stationary (Unit Root)

--- Stationarity Tests for House_Index (1st Difference) ---
ADF Statistic: -3.047
ADF p-value: 0.031
ADF Result: Likely Stationary (reject H0)

KPSS Statistic (c): 0.231
KPSS p-value (c): 0.100
KPSS Result (c): Likely Stationary (fail to reject H0)
Combined: Likely Stationary
Conclusion for House_Index: Likely I(1)
--------------------------------------------------

=============== Testing Variable: CPI ===============

--- Stationarity Tests for CPI (Original) ---
ADF Statistic: 1.356
ADF p-value: 0.997
ADF Result: Likely Non-Stationary (fail to reject H0)

KPSS Statistic (c): 2.090
KPSS p-value (c): 0.010
KPSS Result (c): Likely N